# 05 — VocaLip Thai: 44-Word Real Lip-Reading Extended Training

**Goal:** 44-class mouth-only VSR for live demo, with real mouth-motion gating at deployment.
Uses the frozen clean dataset/cache, exact Experiment 0 architecture, warm-start from the best baseline, max 150 epochs + early stopping.


## Pipeline

1. Environment / imports  
2. Experiment config  
3. Reproducibility  
4. Load clean dataset  
5. Dataset audit  
6. Mouth-cache dataset  
7. Class weights + DataLoaders  
8. Original baseline architecture  
9. Restore best checkpoint  
10. Metrics helpers (NumPy; no scikit-learn required)  
11. Final test inference  
12. Per-class report  
13. Confusion matrix  
14. Confusion-pair analysis  
15. Prediction inspection  
16. Final experiment export  


In [ ]:
# ============================================================
# CELL 1 : ENVIRONMENT / IMPORTS
# ============================================================

import os
import sys
import json
import math
import random
import time
import hashlib
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

print("Python :", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("NumPy  :", np.__version__)
print("Pandas :", pd.__version__)
print("CUDA   :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = DEVICE

print("Device :", DEVICE)
print("scikit-learn required: NO")


In [ ]:
# ============================================================
# CELL 2 : EXPERIMENT CONFIG
# ============================================================
ROOT = Path(r"D:\Face_detection\dataset\word_level_v5\video001")
MANIFEST_PATH = ROOT / "word_train_manifest_v5_baseline_clean.csv"
LABEL_MAP_PATH = ROOT / "word_label_map_v5.json"
CACHE_ROOT = ROOT / "mouth_cache_v3"
BASELINE_BEST_PATH = ROOT / "models_01_baseline" / "01_word_level_baseline_best.pt"
MODEL_DIR = ROOT / "models_05_real_lipreading_44"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODEL_PATH = MODEL_DIR / "05_44word_real_lipreading_best.pt"
LAST_MODEL_PATH = MODEL_DIR / "05_44word_real_lipreading_last.pt"
HISTORY_CSV_PATH = MODEL_DIR / "05_44word_real_lipreading_history.csv"
DEPLOY_MODEL_PATH = MODEL_DIR / "05_44word_real_lipreading_deployment.pt"
SUMMARY_PATH = MODEL_DIR / "05_44word_real_lipreading_summary.json"
NUM_CLASSES=44; TARGET_FRAMES=16; ROI_SIZE=112; CHANNELS=3
FEATURE_DIM=128; GRU_HIDDEN=256; GRU_LAYERS=2; MODEL_DROPOUT=0.20
BATCH_SIZE=8; NUM_WORKERS=0
LR=1e-4; WEIGHT_DECAY=1e-4; EPOCHS=150; PATIENCE=15; GRAD_CLIP_NORM=5.0
USE_CLASS_WEIGHTED_LOSS=True; CLASS_WEIGHT_MIN=0.25; CLASS_WEIGHT_MAX=4.0
WARM_START_FROM_BASELINE=True
print('Manifest:',MANIFEST_PATH); print('Baseline:',BASELINE_BEST_PATH); print('Output:',MODEL_DIR)
assert MANIFEST_PATH.exists(); assert CACHE_ROOT.exists(); assert BASELINE_BEST_PATH.exists()

In [ ]:
# ============================================================
# CELL 3 : REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seed:", SEED)


In [ ]:
# ============================================================
# CELL 4 : LOAD CLEAN DATASET
# ============================================================

df = pd.read_csv(MANIFEST_PATH)

required_columns = {"split", "model_class_id", "word"}
missing_columns = required_columns - set(df.columns)

assert not missing_columns, f"Missing manifest columns: {missing_columns}"

df["model_class_id"] = df["model_class_id"].astype(int)

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df   = df[df["split"] == "val"].reset_index(drop=True)
test_df  = df[df["split"] == "test"].reset_index(drop=True)

class_word_table = (
    df[["model_class_id", "word"]]
    .drop_duplicates()
    .sort_values("model_class_id")
    .reset_index(drop=True)
)

assert len(class_word_table) == NUM_CLASSES

id_to_word = {
    int(row.model_class_id): str(row.word)
    for row in class_word_table.itertuples(index=False)
}
word_to_id = {word: cid for cid, word in id_to_word.items()}

print("=" * 80)
print("CLEAN DATASET")
print("=" * 80)
print("Total :", len(df))
print("Train :", len(train_df))
print("Val   :", len(val_df))
print("Test  :", len(test_df))
print("Classes:", df["model_class_id"].nunique())

assert len(df) == 2245
assert len(train_df) == 1632
assert len(val_df) == 354
assert len(test_df) == 259
assert df["model_class_id"].nunique() == 44

print("\nFirst 10 labels:")
for i in range(10):
    print(f"{i:2d} -> {id_to_word[i]}")


In [ ]:
# ============================================================
# CELL 5 : DATASET AUDIT
# ============================================================

split_summary = (
    df.groupby("split")
      .agg(
          samples=("model_class_id", "size"),
          classes=("model_class_id", "nunique")
      )
      .reset_index()
)

display(split_summary)

train_counts = train_df["model_class_id"].value_counts().sort_index()
val_counts   = val_df["model_class_id"].value_counts().sort_index()
test_counts  = test_df["model_class_id"].value_counts().sort_index()

audit_table = pd.DataFrame({
    "class_id": range(NUM_CLASSES),
    "word": [id_to_word[i] for i in range(NUM_CLASSES)],
    "train": [int(train_counts.get(i, 0)) for i in range(NUM_CLASSES)],
    "val": [int(val_counts.get(i, 0)) for i in range(NUM_CLASSES)],
    "test": [int(test_counts.get(i, 0)) for i in range(NUM_CLASSES)],
})

display(audit_table)

assert (audit_table[["train", "val", "test"]] > 0).all().all()

print("=" * 80)
print("DATASET AUDIT PASSED")
print("=" * 80)
print("Train min/max:", audit_table["train"].min(), audit_table["train"].max())
print("Val   min/max:", audit_table["val"].min(), audit_table["val"].max())
print("Test  min/max:", audit_table["test"].min(), audit_table["test"].max())
print()
print("Scope reminder: current evaluation is IN-DOMAIN and NOT speaker-disjoint.")


In [ ]:
# ============================================================
# CELL 6 : CACHE PATH RESOLVER + DATASET
# ============================================================
IMAGENET_MEAN=torch.tensor([0.485,0.456,0.406],dtype=torch.float32).view(1,3,1,1)
IMAGENET_STD=torch.tensor([0.229,0.224,0.225],dtype=torch.float32).view(1,3,1,1)
def resolve_cache_path(row):
    for key in ['mouth_cache_path','cache_path']:
        if key in row.index and pd.notna(row[key]):
            p=Path(str(row[key]));
            if p.exists(): return p
    for key in ['clip_id','segment_id','sample_id','video_id','filename','clip_name']:
        if key in row.index and pd.notna(row[key]):
            stem=Path(str(row[key])).stem
            for p in [CACHE_ROOT/f'{stem}.npz',CACHE_ROOT/stem/'mouth.npz']:
                if p.exists(): return p
    raise FileNotFoundError(str(row.to_dict()))
class VocaLipWordDataset(Dataset):
    def __init__(self,dataframe,train=False): self.df=dataframe.reset_index(drop=True).copy(); self.train=train
    def __len__(self): return len(self.df)
    def __getitem__(self,idx):
        row=self.df.iloc[idx]; cache_path=resolve_cache_path(row)
        with np.load(cache_path) as data: frames=data['frames'].copy()
        assert frames.shape==(TARGET_FRAMES,ROI_SIZE,ROI_SIZE,3)
        x=torch.from_numpy(frames).float().permute(0,3,1,2)/255.0
        # mild photometric augmentation only; temporal order is preserved
        if self.train and random.random()<0.30:
            x=torch.clamp(x*random.uniform(0.90,1.10)+random.uniform(-0.03,0.03),0,1)
        x=(x-IMAGENET_MEAN)/IMAGENET_STD  # IMPORTANT: cached BGR, NO BGR->RGB swap
        return x,int(row['model_class_id'])
train_dataset=VocaLipWordDataset(train_df,True); val_dataset=VocaLipWordDataset(val_df,False); test_dataset=VocaLipWordDataset(test_df,False)
x0,y0=train_dataset[0]; print(tuple(x0.shape),y0,id_to_word[y0])

In [ ]:
# ============================================================
# CELL 7 : CLASS WEIGHTS + DATALOADERS
# ============================================================
train_counts=train_df['model_class_id'].value_counts().sort_index(); class_weights_np=np.zeros(NUM_CLASSES,np.float32)
for c in range(NUM_CLASSES): class_weights_np[c]=len(train_df)/(NUM_CLASSES*int(train_counts.loc[c]))
class_weights_np=np.clip(class_weights_np,CLASS_WEIGHT_MIN,CLASS_WEIGHT_MAX)
class_weights=torch.tensor(class_weights_np,dtype=torch.float32,device=DEVICE)
pin_memory=torch.cuda.is_available()
train_loader=DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=pin_memory)
val_loader=DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=pin_memory)
test_loader=DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=pin_memory)
print('weights:',class_weights_np.min(),class_weights_np.max(),class_weights_np.mean()); print('batches:',len(train_loader),len(val_loader),len(test_loader))

In [ ]:
# ============================================================
# CELL 8 : ORIGINAL BASELINE MODEL — EXACT RECOVERY VERSION
# ============================================================

class FrameCNNEncoder(nn.Module):
    def __init__(self, feature_dim=128):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 96, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(96),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(96, 128, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        # IMPORTANT:
        # Original Experiment 0 included ReLU + Dropout after projection.
        # These layers have no checkpoint parameters, so strict=True can still
        # succeed even if they are accidentally omitted. ReLU DOES change inference.
        self.projection = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, feature_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.10),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.projection(x)
        return x


class TemporalAttention(nn.Module):
    def __init__(self, input_dim, attention_dim=128):
        super().__init__()

        self.attention = nn.Sequential(
            nn.Linear(input_dim, attention_dim),
            nn.Tanh(),
            nn.Linear(attention_dim, 1),
        )

    def forward(self, x):
        scores = self.attention(x).squeeze(-1)
        weights = torch.softmax(scores, dim=1)
        pooled = torch.sum(x * weights.unsqueeze(-1), dim=1)
        return pooled, weights


class VocaLipWordBaseline(nn.Module):
    def __init__(
        self,
        num_classes,
        feature_dim=128,
        gru_hidden=256,
        gru_layers=2,
        dropout=0.20,
    ):
        super().__init__()

        self.frame_encoder = FrameCNNEncoder(feature_dim=feature_dim)

        self.temporal_encoder = nn.GRU(
            input_size=feature_dim,
            hidden_size=gru_hidden,
            num_layers=gru_layers,
            batch_first=True,
            bidirectional=True,
            dropout=(dropout if gru_layers > 1 else 0.0),
        )

        temporal_dim = gru_hidden * 2

        self.temporal_attention = TemporalAttention(
            input_dim=temporal_dim,
            attention_dim=128,
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(temporal_dim),
            nn.Dropout(p=dropout),
            nn.Linear(temporal_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, video, return_attention=False):
        B, T, C, H, W = video.shape

        x = video.reshape(B * T, C, H, W)
        frame_features = self.frame_encoder(x)
        frame_features = frame_features.reshape(B, T, -1)

        temporal_features, _ = self.temporal_encoder(frame_features)
        pooled, attention_weights = self.temporal_attention(temporal_features)

        logits = self.classifier(pooled)

        if return_attention:
            return logits, attention_weights

        return logits


model = VocaLipWordBaseline(
    num_classes=NUM_CLASSES,
    feature_dim=FEATURE_DIM,
    gru_hidden=GRU_HIDDEN,
    gru_layers=GRU_LAYERS,
    dropout=MODEL_DROPOUT,
).to(DEVICE)

total_parameters = sum(p.numel() for p in model.parameters())

print("=" * 80)
print("ORIGINAL BASELINE ARCHITECTURE")
print("=" * 80)
print("Parameters:", f"{total_parameters:,}")
print("Projection : Linear -> ReLU -> Dropout(0.10)")
print("Temporal   : 2-layer BiGRU")
print("Attention  : 512 -> 128 -> 1")
print("Classes    :", NUM_CLASSES)

assert total_parameters == 2_187_405

print("✅ Exact Experiment 0 architecture recreated.")


In [ ]:
# CELL 9 : WARM START FROM FROZEN EXPERIMENT 0
if WARM_START_FROM_BASELINE:
    ckpt=torch.load(BASELINE_BEST_PATH,map_location=DEVICE,weights_only=False)
    state=ckpt.get('model_state_dict',ckpt.get('state_dict',ckpt))
    model.load_state_dict(state,strict=True)
    print('Warm-start strict load: OK | source epoch:',ckpt.get('epoch'))
else: print('Random initialization')

In [ ]:
# CELL 10 : LOSS / OPTIMIZER / SCHEDULER / AMP
criterion=nn.CrossEntropyLoss(weight=class_weights)
optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='max',factor=0.5,patience=4,min_lr=1e-6)
scaler=torch.amp.GradScaler('cuda',enabled=torch.cuda.is_available())
print('LR',LR,'AMP',scaler.is_enabled())

In [ ]:
# CELL 11 : METRICS + TRAIN/EVAL (NO SCIKIT-LEARN)
def metrics_np(y,p,probs):
    y=np.asarray(y); p=np.asarray(p); cm=np.zeros((NUM_CLASSES,NUM_CLASSES),dtype=np.int64)
    for a,b in zip(y,p): cm[a,b]+=1
    f=[]
    for c in range(NUM_CLASSES):
        tp=cm[c,c]; fp=cm[:,c].sum()-tp; fn=cm[c,:].sum()-tp
        pr=tp/(tp+fp) if tp+fp else 0.; rc=tp/(tp+fn) if tp+fn else 0.
        f.append(2*pr*rc/(pr+rc) if pr+rc else 0.)
    top3=np.argsort(probs,axis=1)[:,-3:]
    return float((y==p).mean()),float(np.mean(f)),float(np.mean([a in r for a,r in zip(y,top3)]))
def run_epoch(model,loader,training=False):
    model.train(training); losses=[]; ys=[]; ps=[]; prob_all=[]
    for x,y in tqdm(loader,leave=False):
        x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True)
        if training: optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda',enabled=torch.cuda.is_available()): logits=model(x); loss=criterion(logits,y)
        if training:
            scaler.scale(loss).backward(); scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP_NORM); scaler.step(optimizer); scaler.update()
        prob=torch.softmax(logits.detach(),1).float().cpu().numpy(); pred=prob.argmax(1)
        losses.append(loss.item()); ys.extend(y.cpu().tolist()); ps.extend(pred.tolist()); prob_all.append(prob)
    probs=np.concatenate(prob_all); acc,f1,t3=metrics_np(ys,ps,probs)
    return {'loss':float(np.mean(losses)),'accuracy':acc,'macro_f1':f1,'top3':t3,'y_true':np.array(ys),'y_pred':np.array(ps),'probs':probs}

In [ ]:
# CELL 12 : SANITY FORWARD
model.eval(); xb,yb=next(iter(val_loader)); xb=xb[:2].to(DEVICE)
with torch.inference_mode(),torch.amp.autocast('cuda',enabled=torch.cuda.is_available()): logits,attn=model(xb,return_attention=True)
print('input',tuple(xb.shape),'logits',tuple(logits.shape),'attn',tuple(attn.shape)); assert logits.shape==(2,44); print('SANITY PASS')

In [ ]:
# CELL 13 : EXTENDED TRAINING — MAX 150, EARLY STOP 15
history=[]; best_f1=-1.; best_epoch=0; bad=0; start=time.time()
for epoch in range(1,EPOCHS+1):
    tr=run_epoch(model,train_loader,True)
    with torch.no_grad(): va=run_epoch(model,val_loader,False)
    scheduler.step(va['macro_f1']); lr_now=optimizer.param_groups[0]['lr']
    row={'epoch':epoch,'lr':lr_now,'train_loss':tr['loss'],'train_acc':tr['accuracy'],'train_f1':tr['macro_f1'],'train_top3':tr['top3'],'val_loss':va['loss'],'val_acc':va['accuracy'],'val_f1':va['macro_f1'],'val_top3':va['top3']}; history.append(row); pd.DataFrame(history).to_csv(HISTORY_CSV_PATH,index=False)
    if va['macro_f1']>best_f1+1e-6:
        best_f1=va['macro_f1']; best_epoch=epoch; bad=0
        torch.save({'epoch':epoch,'model_state_dict':model.state_dict(),'val_metrics':va,'config':{'num_classes':44,'target_frames':16,'roi_size':112,'feature_dim':128,'gru_hidden':256,'gru_layers':2,'dropout':.20}},BEST_MODEL_PATH)
    else: bad+=1
    torch.save({'epoch':epoch,'model_state_dict':model.state_dict()},LAST_MODEL_PATH)
    print(f"E{epoch:03d} train acc={tr['accuracy']:.3f} f1={tr['macro_f1']:.3f} | val acc={va['accuracy']:.3f} f1={va['macro_f1']:.3f} top3={va['top3']:.3f} | lr={lr_now:.2e} | best E{best_epoch}={best_f1:.4f}")
    if bad>=PATIENCE: print('EARLY STOP'); break
print('Done | best epoch',best_epoch,'best val F1',best_f1,'minutes',(time.time()-start)/60)

In [ ]:
# CELL 14 : LOAD BEST + FINAL TEST (RUN ONCE AFTER TRAINING)
best=torch.load(BEST_MODEL_PATH,map_location=DEVICE,weights_only=False); model.load_state_dict(best['model_state_dict'],strict=True); model.eval()
with torch.no_grad(): test=run_epoch(model,test_loader,False)
print('FINAL TEST | acc',f"{test['accuracy']*100:.2f}%",'F1',f"{test['macro_f1']:.4f}",'Top3',f"{test['top3']*100:.2f}%",'best epoch',best['epoch'])

In [ ]:
# CELL 15 : PER-CLASS REPORT — USE THIS TO CHOOSE WORDS TO REHEARSE FOR DEMO
y=test['y_true']; p=test['y_pred']; cm=np.zeros((44,44),dtype=np.int64)
for a,b in zip(y,p): cm[a,b]+=1
rows=[]
for c in range(44):
    tp=cm[c,c]; sup=cm[c].sum(); pc=cm[:,c].sum(); pr=tp/pc if pc else 0.; rc=tp/sup if sup else 0.; f=2*pr*rc/(pr+rc) if pr+rc else 0.
    rows.append({'class_id':c,'word':id_to_word[c],'support':int(sup),'correct':int(tp),'precision':pr,'recall':rc,'f1':f})
report=pd.DataFrame(rows).sort_values(['f1','recall','support'],ascending=False); display(report.head(20)); report.to_csv(MODEL_DIR/'05_44word_test_class_report.csv',index=False,encoding='utf-8-sig')

In [ ]:
# CELL 16 : MOUTH-MOTION GATE CALIBRATION
# Gate prevents predictions while the mouth is idle. It does NOT replace the 44-word classifier.
def motion_score(frames):
    x=frames.astype(np.float32)/255.; g=x.mean(-1); d=np.abs(g[1:]-g[:-1]); h,w=g.shape[1:]; d=d[:,int(.25*h):int(.80*h),int(.12*w):int(.88*w)]; return float(d.mean())
scores=[]
for _,row in tqdm(val_df.iterrows(),total=len(val_df),desc='motion'):
    with np.load(resolve_cache_path(row)) as z: frames=z['frames']
    scores.append(motion_score(frames))
scores=np.asarray(scores); MOTION_THRESHOLD_START=float(np.percentile(scores,10))
print('speech motion min/p10/median/p90/max:',float(scores.min()),MOTION_THRESHOLD_START,float(np.median(scores)),float(np.percentile(scores,90)),float(scores.max()))
print('IMPORTANT: before live demo, measure 20-30 sec idle webcam motion and set threshold ABOVE idle noise but below normal speech motion.')

In [ ]:
# CELL 17 : EXPORT DEPLOYMENT CHECKPOINT
CONFIDENCE_THRESHOLD_START=0.35; MARGIN_THRESHOLD_START=0.08
deploy={'format':'VocaLip-44word-real-lipreading-v1','model_state_dict':model.state_dict(),'model_config':{'num_classes':44,'feature_dim':128,'gru_hidden':256,'gru_layers':2,'dropout':.20},'preprocessing':{'target_frames':16,'roi_size':112,'cache_channel_order':'BGR','convert_bgr_to_rgb':False,'mean':[.485,.456,.406],'std':[.229,.224,.225],'input_layout':'B,T,C,H,W'},'motion_gate':{'enabled':True,'starting_threshold':MOTION_THRESHOLD_START,'rule':'Do not create a new word prediction while mouth motion is below threshold'},'rejection':{'confidence_threshold_start':CONFIDENCE_THRESHOLD_START,'top1_top2_margin_start':MARGIN_THRESHOLD_START,'reject_text':'ไม่แน่ใจ กรุณาพูดอีกครั้ง'},'id_to_word':id_to_word,'word_to_id':word_to_id,'test_metrics':{'accuracy':test['accuracy'],'macro_f1':test['macro_f1'],'top3':test['top3']},'scope':'44-word closed-vocabulary research/demo; current split is in-domain and not speaker-disjoint'}
torch.save(deploy,DEPLOY_MODEL_PATH)
summary={'best_epoch':int(best['epoch']),'test_accuracy':test['accuracy'],'test_macro_f1':test['macro_f1'],'test_top3':test['top3'],'motion_threshold_start':MOTION_THRESHOLD_START,'checkpoint':str(DEPLOY_MODEL_PATH)}
with open(SUMMARY_PATH,'w',encoding='utf8') as f: json.dump(summary,f,ensure_ascii=False,indent=2)
print('DEPLOYMENT READY:',DEPLOY_MODEL_PATH)

In [ ]:
# ============================================================
# TRAINING HISTORY GRAPHS — VOCALIP 44 WORDS
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path(r"D:\Face_detection\dataset\word_level_v5\video001")

HISTORY_PATH = (
    ROOT
    / "models_05_real_lipreading_44"
    / "05_44word_real_lipreading_history.csv"
)

SAVE_DIR = ROOT / "models_05_real_lipreading_44" / "graphs"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# LOAD HISTORY
# ------------------------------------------------------------

history = pd.read_csv(HISTORY_PATH)

print("Epochs:", len(history))
print(history.columns.tolist())

# Best epoch = highest Validation Macro-F1
best_idx = history["val_f1"].idxmax()
best_epoch = int(history.loc[best_idx, "epoch"])
best_val_f1 = float(history.loc[best_idx, "val_f1"])

print(f"Best Epoch    : {best_epoch}")
print(f"Best Val F1   : {best_val_f1:.4f}")


# ============================================================
# GRAPH 1 — TRAIN vs VALIDATION ACCURACY
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(
    history["epoch"],
    history["train_acc"],
    label="Train Accuracy",
    linewidth=2
)

plt.plot(
    history["epoch"],
    history["val_acc"],
    label="Validation Accuracy",
    linewidth=2
)

plt.axvline(
    best_epoch,
    linestyle="--",
    alpha=0.7,
    label=f"Best Epoch = {best_epoch}"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("VocaLip 44-Word Model — Training & Validation Accuracy")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()

path = SAVE_DIR / "01_accuracy_curve.png"
plt.savefig(path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", path)


# ============================================================
# GRAPH 2 — TRAIN vs VALIDATION MACRO-F1
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(
    history["epoch"],
    history["train_f1"],
    label="Train Macro-F1",
    linewidth=2
)

plt.plot(
    history["epoch"],
    history["val_f1"],
    label="Validation Macro-F1",
    linewidth=2
)

plt.scatter(
    best_epoch,
    best_val_f1,
    s=100,
    zorder=5,
    label=f"Best F1 = {best_val_f1:.4f}"
)

plt.axvline(
    best_epoch,
    linestyle="--",
    alpha=0.7
)

plt.xlabel("Epoch")
plt.ylabel("Macro-F1")
plt.title("VocaLip 44-Word Model — Macro-F1")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()

path = SAVE_DIR / "02_macro_f1_curve.png"
plt.savefig(path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", path)


# ============================================================
# GRAPH 3 — TRAIN vs VALIDATION LOSS
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(
    history["epoch"],
    history["train_loss"],
    label="Train Loss",
    linewidth=2
)

plt.plot(
    history["epoch"],
    history["val_loss"],
    label="Validation Loss",
    linewidth=2
)

plt.axvline(
    best_epoch,
    linestyle="--",
    alpha=0.7,
    label=f"Best Epoch = {best_epoch}"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("VocaLip 44-Word Model — Training & Validation Loss")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()

path = SAVE_DIR / "03_loss_curve.png"
plt.savefig(path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", path)


# ============================================================
# GRAPH 4 — TOP-3 ACCURACY
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(
    history["epoch"],
    history["train_top3"],
    label="Train Top-3",
    linewidth=2
)

plt.plot(
    history["epoch"],
    history["val_top3"],
    label="Validation Top-3",
    linewidth=2
)

plt.axvline(
    best_epoch,
    linestyle="--",
    alpha=0.7,
    label=f"Best Epoch = {best_epoch}"
)

plt.xlabel("Epoch")
plt.ylabel("Top-3 Accuracy")
plt.title("VocaLip 44-Word Model — Top-3 Accuracy")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()

path = SAVE_DIR / "04_top3_curve.png"
plt.savefig(path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", path)


# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("TRAINING GRAPH EXPORT COMPLETE")
print("=" * 70)

print(f"Total epochs : {len(history)}")
print(f"Best epoch   : {best_epoch}")
print(f"Best Val F1  : {best_val_f1:.4f}")

print("\nGraphs saved to:")
print(SAVE_DIR)

In [ ]:
# ============================================================
# MODEL COMPARISON — BASELINE vs SSL vs FINAL MODEL
# ============================================================

import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path(r"D:\Face_detection\dataset\word_level_v5\video001")

# ============================================================
# RESULTS FROM PREVIOUS EXPERIMENTS
# ============================================================

experiments = {
    "Baseline\nSupervised": {
        "accuracy": 0.2625,
        "macro_f1": 0.2327,
        "top3": 0.4826,
    },

    "SSL +\nFine-tuning": {
        "accuracy": 0.2085,
        "macro_f1": 0.1798,
        "top3": None,   # ถ้าไม่มีผลที่ยืนยันไว้ จะไม่เดา
    },
}

# ============================================================
# LOAD FINAL MODEL RESULT AUTOMATICALLY
# ============================================================

SUMMARY_PATH = (
    ROOT
    / "models_05_real_lipreading_44"
    / "05_44word_real_lipreading_summary.json"
)

if SUMMARY_PATH.exists():

    with open(SUMMARY_PATH, "r", encoding="utf-8") as f:
        final_result = json.load(f)

    experiments["Final Model\n44 Words"] = {
        "accuracy": final_result["test_accuracy"],
        "macro_f1": final_result["test_macro_f1"],
        "top3": final_result["test_top3"],
    }

    print("Loaded Final Model result:")
    print(final_result)

else:
    print("WARNING: Final summary not found")
    print(SUMMARY_PATH)

# ============================================================
# SHOW RESULTS
# ============================================================

print("\nMODEL COMPARISON")
print("=" * 70)

for name, result in experiments.items():

    print(name.replace("\n", " "))

    print(
        f" Accuracy : "
        f"{result['accuracy']*100:.2f}%"
    )

    print(
        f" Macro-F1 : "
        f"{result['macro_f1']:.4f}"
    )

    if result["top3"] is not None:
        print(
            f" Top-3    : "
            f"{result['top3']*100:.2f}%"
        )

    print()

# ============================================================
# PREPARE DATA
# ============================================================

names = list(experiments.keys())

accuracy = [
    experiments[n]["accuracy"] * 100
    for n in names
]

macro_f1 = [
    experiments[n]["macro_f1"] * 100
    for n in names
]

top3 = [
    (
        experiments[n]["top3"] * 100
        if experiments[n]["top3"] is not None
        else np.nan
    )
    for n in names
]

x = np.arange(len(names))

width = 0.25

# ============================================================
# GRAPH 1 — OVERALL MODEL COMPARISON
# ============================================================

fig, ax = plt.subplots(figsize=(12, 7))

bars1 = ax.bar(
    x - width,
    accuracy,
    width,
    label="Top-1 Accuracy"
)

bars2 = ax.bar(
    x,
    macro_f1,
    width,
    label="Macro-F1"
)

bars3 = ax.bar(
    x + width,
    top3,
    width,
    label="Top-3 Accuracy"
)

ax.set_ylabel("Score (%)")

ax.set_title(
    "VocaLip Thai — Model Performance Comparison"
)

ax.set_xticks(x)
ax.set_xticklabels(names)

ax.set_ylim(0, 100)

ax.legend()

ax.grid(
    axis="y",
    alpha=0.25
)

# ============================================================
# ADD VALUES ABOVE BARS
# ============================================================

def add_labels(bars):

    for bar in bars:

        height = bar.get_height()

        if np.isnan(height):
            continue

        ax.annotate(
            f"{height:.1f}%",
            xy=(
                bar.get_x() + bar.get_width()/2,
                height
            ),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=10,
            fontweight="bold"
        )

add_labels(bars1)
add_labels(bars2)
add_labels(bars3)

plt.tight_layout()

SAVE_DIR = (
    ROOT
    / "models_05_real_lipreading_44"
    / "graphs"
)

SAVE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT = (
    SAVE_DIR
    / "05_model_comparison.png"
)

plt.savefig(
    OUTPUT,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved:", OUTPUT)

# ============================================================
# GRAPH 2 — ACCURACY ONLY
# Better for presentation slide
# ============================================================

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.bar(
    names,
    accuracy
)

ax.set_ylabel("Test Accuracy (%)")

ax.set_title(
    "VocaLip Thai — Test Accuracy by Experiment"
)

ax.set_ylim(
    0,
    max(40, max(accuracy) + 10)
)

ax.grid(
    axis="y",
    alpha=0.25
)

for bar, value in zip(bars, accuracy):

    ax.text(
        bar.get_x() + bar.get_width()/2,
        value + 0.7,
        f"{value:.2f}%",
        ha="center",
        fontweight="bold"
    )

plt.tight_layout()

OUTPUT2 = (
    SAVE_DIR
    / "06_accuracy_comparison.png"
)

plt.savefig(
    OUTPUT2,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved:", OUTPUT2)

In [ ]:
# ============================================================
# CONFUSION MATRIX RECOVERY — VOCALIP 44 WORDS
# โหลด Best Model -> Test Inference -> Confusion Matrix
# ไม่ต้อง Train ใหม่
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import torch
from pathlib import Path
from tqdm.auto import tqdm

# ------------------------------------------------------------
# CHECK REQUIRED VARIABLES
# ------------------------------------------------------------

required = [
    "model",
    "test_loader",
    "DEVICE",
    "id_to_word",
]

missing = [
    name for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "ยังขาดตัวแปร: "
        + ", ".join(missing)
        + "\nให้รัน Cell 1-13 ของ Notebook 05 ก่อน "
          "(ไม่ต้องรัน Training Cell 14)"
    )

# ------------------------------------------------------------
# PATH
# ------------------------------------------------------------

ROOT = Path(
    r"D:\Face_detection\dataset\word_level_v5\video001"
)

BEST_MODEL_PATH = (
    ROOT
    / "models_05_real_lipreading_44"
    / "05_44word_real_lipreading_best.pt"
)

SAVE_DIR = (
    ROOT
    / "models_05_real_lipreading_44"
    / "graphs"
)

SAVE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

assert BEST_MODEL_PATH.exists(), (
    f"ไม่พบ Best Model:\n{BEST_MODEL_PATH}"
)

print("Loading:")
print(BEST_MODEL_PATH)

# ============================================================
# LOAD BEST MODEL
# ============================================================

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE,
    weights_only=False
)

if "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
else:
    state_dict = checkpoint

model.load_state_dict(
    state_dict,
    strict=True
)

model.to(DEVICE)
model.eval()

print("✅ Best model loaded")

if isinstance(checkpoint, dict):
    print(
        "Best epoch:",
        checkpoint.get("epoch", "unknown")
    )

# ============================================================
# RUN TEST INFERENCE
# ============================================================

all_true = []
all_pred = []
all_probs = []

print("\nRunning test inference...")

with torch.inference_mode():

    for batch in tqdm(
        test_loader,
        desc="Test inference"
    ):

        x = batch["video"].to(
            DEVICE,
            non_blocking=True
        )

        y = batch["label"].to(
            DEVICE,
            non_blocking=True
        )

        # ใช้ AMP แบบเดียวกับ evaluation เดิม
        with torch.amp.autocast(
            "cuda",
            enabled=(DEVICE.type == "cuda")
        ):
            logits = model(x)

        probs = torch.softmax(
            logits.float(),
            dim=1
        )

        pred = probs.argmax(dim=1)

        all_true.extend(
            y.cpu().numpy()
        )

        all_pred.extend(
            pred.cpu().numpy()
        )

        all_probs.append(
            probs.cpu().numpy()
        )

# ------------------------------------------------------------
# CREATE ARRAYS
# ------------------------------------------------------------

y_true = np.asarray(
    all_true,
    dtype=np.int64
)

y_pred = np.asarray(
    all_pred,
    dtype=np.int64
)

test_probs = np.concatenate(
    all_probs,
    axis=0
)

print("\n✅ Test inference complete")
print("Samples:", len(y_true))

# ============================================================
# BASIC ACCURACY
# ============================================================

accuracy = (
    y_true == y_pred
).mean()

print(
    f"Accuracy: "
    f"{accuracy*100:.2f}%"
)

# ============================================================
# CONFUSION MATRIX
# ============================================================

NUM_CLASSES = 44

cm = np.zeros(
    (NUM_CLASSES, NUM_CLASSES),
    dtype=np.int64
)

for true_id, pred_id in zip(
    y_true,
    y_pred
):

    cm[
        int(true_id),
        int(pred_id)
    ] += 1

# ============================================================
# NORMALIZE EACH TRUE CLASS
# ============================================================

row_sum = cm.sum(
    axis=1,
    keepdims=True
)

cm_norm = np.divide(
    cm,
    row_sum,
    out=np.zeros_like(
        cm,
        dtype=np.float32
    ),
    where=row_sum != 0
)

# ------------------------------------------------------------
# CLASS NAMES
# ------------------------------------------------------------

class_names = [
    id_to_word.get(i, str(i))
    for i in range(NUM_CLASSES)
]

# ============================================================
# PLOT NORMALIZED CONFUSION MATRIX
# ============================================================

fig, ax = plt.subplots(
    figsize=(22, 20)
)

im = ax.imshow(
    cm_norm,
    interpolation="nearest",
    cmap="Blues",
    vmin=0,
    vmax=1
)

cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.046,
    pad=0.04
)

cbar.set_label(
    "Normalized prediction rate"
)

ax.set_xticks(
    np.arange(NUM_CLASSES)
)

ax.set_yticks(
    np.arange(NUM_CLASSES)
)

ax.set_xticklabels(
    class_names,
    rotation=90,
    fontsize=9
)

ax.set_yticklabels(
    class_names,
    fontsize=9
)

ax.set_xlabel(
    "Predicted Word",
    fontsize=14
)

ax.set_ylabel(
    "True Word",
    fontsize=14
)

ax.set_title(
    "VocaLip Thai — Normalized Confusion Matrix (44 Words)",
    fontsize=17,
    pad=20
)

plt.tight_layout()

OUTPUT = (
    SAVE_DIR
    / "07_confusion_matrix_normalized.png"
)

plt.savefig(
    OUTPUT,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nSaved:")
print(OUTPUT)

# ============================================================
# TOP CONFUSIONS
# ============================================================

confusions = []

for true_id in range(NUM_CLASSES):

    support = cm[
        true_id
    ].sum()

    for pred_id in range(NUM_CLASSES):

        if true_id == pred_id:
            continue

        count = int(
            cm[
                true_id,
                pred_id
            ]
        )

        if count == 0:
            continue

        rate = (
            count / support
            if support > 0
            else 0
        )

        confusions.append(
            (
                count,
                rate,
                true_id,
                pred_id
            )
        )

confusions.sort(
    reverse=True
)

print("\n" + "=" * 75)
print("TOP CONFUSED WORD PAIRS")
print("=" * 75)

for rank, (
    count,
    rate,
    true_id,
    pred_id
) in enumerate(
    confusions[:20],
    start=1
):

    print(
        f"{rank:02d}. "
        f"จริง: {class_names[true_id]:<12} "
        f"→ ทาย: {class_names[pred_id]:<12} "
        f"| {count} ครั้ง "
        f"| {rate*100:.1f}%"
    )

print("\n✅ CONFUSION MATRIX COMPLETE")

In [ ]:
batch = next(iter(test_loader))

print("Batch type:", type(batch))

if isinstance(batch, (list, tuple)):
    print("Batch length:", len(batch))

    for i, item in enumerate(batch):
        print(
            f"[{i}]",
            type(item),
            getattr(item, "shape", None)
        )

elif isinstance(batch, dict):
    print("Keys:", batch.keys())

    for k, v in batch.items():
        print(
            k,
            type(v),
            getattr(v, "shape", None)
        )

In [ ]:
# ============================================================
# ROBUST TEST INFERENCE
# รองรับทั้ง dict / list / tuple
# ============================================================

all_true = []
all_pred = []
all_probs = []

model.eval()

print("Running test inference...")

with torch.inference_mode():

    for batch in tqdm(
        test_loader,
        desc="Test inference"
    ):

        # ----------------------------------------------------
        # CASE 1 : Dictionary dataset
        # ----------------------------------------------------

        if isinstance(batch, dict):

            x = batch["video"]
            y = batch["label"]

        # ----------------------------------------------------
        # CASE 2 : Tuple / List dataset
        # ----------------------------------------------------

        elif isinstance(batch, (list, tuple)):

            if len(batch) < 2:
                raise RuntimeError(
                    f"Batch มีเพียง {len(batch)} elements "
                    "จึงหา video + label ไม่ได้"
                )

            x = batch[0]
            y = batch[1]

        else:

            raise TypeError(
                f"Unknown batch type: {type(batch)}"
            )

        # ----------------------------------------------------
        # MOVE TO GPU
        # ----------------------------------------------------

        x = x.to(
            DEVICE,
            non_blocking=True
        )

        y = y.to(
            DEVICE,
            non_blocking=True
        )

        # ----------------------------------------------------
        # MODEL INFERENCE
        # ----------------------------------------------------

        with torch.amp.autocast(
            "cuda",
            enabled=(DEVICE.type == "cuda")
        ):

            logits = model(x)

        # ----------------------------------------------------
        # SOFTMAX
        # ----------------------------------------------------

        probs = torch.softmax(
            logits.float(),
            dim=1
        )

        pred = probs.argmax(
            dim=1
        )

        # ----------------------------------------------------
        # SAVE RESULTS
        # ----------------------------------------------------

        all_true.extend(
            y.detach().cpu().numpy().tolist()
        )

        all_pred.extend(
            pred.detach().cpu().numpy().tolist()
        )

        all_probs.append(
            probs.detach().cpu().numpy()
        )


# ============================================================
# CREATE RESULT ARRAYS
# ============================================================

y_true = np.asarray(
    all_true,
    dtype=np.int64
)

y_pred = np.asarray(
    all_pred,
    dtype=np.int64
)

test_probs = np.concatenate(
    all_probs,
    axis=0
)

# ============================================================
# QUICK CHECK
# ============================================================

accuracy = np.mean(
    y_true == y_pred
)

print("\n" + "=" * 60)
print("TEST INFERENCE COMPLETE")
print("=" * 60)

print("Samples :", len(y_true))
print("Correct :", int((y_true == y_pred).sum()))
print("Wrong   :", int((y_true != y_pred).sum()))

print(
    f"Accuracy: {accuracy*100:.2f}%"
)

print("Probability shape:", test_probs.shape)

In [ ]:
# ============================================================
# CONFUSION MATRIX — FINAL MODEL 44 WORDS
# ใช้ y_true / y_pred ที่ได้จาก Test inference ล่าสุด
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

NUM_CLASSES = 44

# ------------------------------------------------------------
# CHECK
# ------------------------------------------------------------

assert "y_true" in globals(), "ไม่พบ y_true"
assert "y_pred" in globals(), "ไม่พบ y_pred"

assert len(y_true) == len(y_pred)

print("Samples:", len(y_true))
print(
    "Accuracy:",
    f"{np.mean(y_true == y_pred) * 100:.2f}%"
)

# ------------------------------------------------------------
# CLASS NAMES
# ------------------------------------------------------------

class_names = [
    id_to_word.get(i, str(i))
    for i in range(NUM_CLASSES)
]

# ------------------------------------------------------------
# BUILD CONFUSION MATRIX
# ------------------------------------------------------------

cm = np.zeros(
    (NUM_CLASSES, NUM_CLASSES),
    dtype=np.int64
)

for true_id, pred_id in zip(y_true, y_pred):
    cm[int(true_id), int(pred_id)] += 1

# ------------------------------------------------------------
# NORMALIZED MATRIX
# ------------------------------------------------------------

row_sum = cm.sum(axis=1, keepdims=True)

cm_norm = np.divide(
    cm,
    row_sum,
    out=np.zeros_like(cm, dtype=float),
    where=row_sum != 0
)

# ------------------------------------------------------------
# SAVE DIRECTORY
# ------------------------------------------------------------

ROOT = Path(
    r"D:\Face_detection\dataset\word_level_v5\video001"
)

SAVE_DIR = (
    ROOT
    / "models_05_real_lipreading_44"
    / "graphs"
)

SAVE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# PLOT
# ============================================================

fig, ax = plt.subplots(figsize=(22, 20))

im = ax.imshow(
    cm_norm,
    interpolation="nearest",
    cmap="Blues",
    vmin=0,
    vmax=1
)

cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.046,
    pad=0.04
)

cbar.set_label(
    "Normalized Prediction Rate",
    fontsize=12
)

# ------------------------------------------------------------
# LABELS
# ------------------------------------------------------------

ax.set_xticks(np.arange(NUM_CLASSES))
ax.set_yticks(np.arange(NUM_CLASSES))

ax.set_xticklabels(
    class_names,
    rotation=90,
    fontsize=9
)

ax.set_yticklabels(
    class_names,
    fontsize=9
)

ax.set_xlabel(
    "Predicted Word",
    fontsize=14
)

ax.set_ylabel(
    "True Word",
    fontsize=14
)

ax.set_title(
    "VocaLip Thai — Normalized Confusion Matrix (44 Words)",
    fontsize=17,
    pad=20
)

# ------------------------------------------------------------
# GRID
# ------------------------------------------------------------

ax.set_xticks(
    np.arange(-0.5, NUM_CLASSES, 1),
    minor=True
)

ax.set_yticks(
    np.arange(-0.5, NUM_CLASSES, 1),
    minor=True
)

ax.grid(
    which="minor",
    linewidth=0.2,
    alpha=0.3
)

ax.tick_params(
    which="minor",
    bottom=False,
    left=False
)

plt.tight_layout()

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

OUTPUT = (
    SAVE_DIR
    / "07_confusion_matrix_normalized.png"
)

plt.savefig(
    OUTPUT,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nSaved:")
print(OUTPUT)

# ============================================================
# TOP CONFUSIONS
# ============================================================

confusions = []

for true_id in range(NUM_CLASSES):

    support = int(cm[true_id].sum())

    for pred_id in range(NUM_CLASSES):

        if true_id == pred_id:
            continue

        count = int(
            cm[true_id, pred_id]
        )

        if count == 0:
            continue

        rate = (
            count / support
            if support > 0
            else 0
        )

        confusions.append({
            "true_word": class_names[true_id],
            "pred_word": class_names[pred_id],
            "count": count,
            "rate": rate
        })

confusions = sorted(
    confusions,
    key=lambda x: (
        x["count"],
        x["rate"]
    ),
    reverse=True
)

print("\n" + "=" * 70)
print("TOP CONFUSED WORD PAIRS")
print("=" * 70)

for i, item in enumerate(
    confusions[:20],
    start=1
):

    print(
        f"{i:02d}. "
        f"{item['true_word']} "
        f"→ {item['pred_word']} "
        f"| {item['count']} ครั้ง "
        f"| {item['rate']*100:.1f}%"
    )

In [ ]:
import pandas as pd
from pathlib import Path

ROOT = Path(r"D:\Face_detection\dataset\word_level_v5\video001")

HISTORY_PATH = (
    ROOT
    / "models_05_real_lipreading_44"
    / "05_44word_real_lipreading_history.csv"
)

df = pd.read_csv(HISTORY_PATH)

print("=" * 60)
print("TABLE 4.2 — TRAINING RESULTS")
print("=" * 60)

print(f"จำนวน Epoch ที่ใช้ในการฝึก      : {len(df)} Epoch")
print(f"Training Loss ต่ำสุด            : {df['train_loss'].min():.6f}")
print(f"Validation Loss ต่ำสุด          : {df['val_loss'].min():.6f}")
print(f"Training Accuracy สูงสุด        : {df['train_acc'].max()*100:.2f}%")
print(f"Validation Accuracy สูงสุด      : {df['val_acc'].max()*100:.2f}%")

best_idx = df["val_f1"].idxmax()

print(f"Epoch ที่ได้แบบจำลองที่ดีที่สุด : {int(df.loc[best_idx, 'epoch'])}")
print(f"Best Validation Macro-F1        : {df.loc[best_idx, 'val_f1']:.4f}")
print(f"Best Epoch Validation Accuracy  : {df.loc[best_idx, 'val_acc']*100:.2f}%")
print(f"Best Epoch Top-3 Accuracy       : {df.loc[best_idx, 'val_top3']*100:.2f}%")

In [ ]:
# ============================================================
# TABLE 4.3 — FINAL TEST METRICS
# ไม่ใช้ sklearn
# ============================================================

import numpy as np

NUM_CLASSES = 44

# ------------------------------------------------------------
# CHECK
# ------------------------------------------------------------
assert "y_true" in globals(), "ไม่พบ y_true"
assert "y_pred" in globals(), "ไม่พบ y_pred"
assert "test_probs" in globals(), "ไม่พบ test_probs"

y_true = np.asarray(y_true, dtype=np.int64)
y_pred = np.asarray(y_pred, dtype=np.int64)
test_probs = np.asarray(test_probs)

# ------------------------------------------------------------
# CONFUSION MATRIX
# ------------------------------------------------------------
cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)

for t, p in zip(y_true, y_pred):
    cm[t, p] += 1

# ------------------------------------------------------------
# PER-CLASS METRICS
# ------------------------------------------------------------
precision_list = []
recall_list = []
f1_list = []
support_list = []

for c in range(NUM_CLASSES):

    TP = cm[c, c]
    FP = cm[:, c].sum() - TP
    FN = cm[c, :].sum() - TP

    support = cm[c, :].sum()

    precision = (
        TP / (TP + FP)
        if (TP + FP) > 0 else 0.0
    )

    recall = (
        TP / (TP + FN)
        if (TP + FN) > 0 else 0.0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0 else 0.0
    )

    precision_list.append(precision)
    recall_list.append(recall)
    f1_list.append(f1)
    support_list.append(support)

precision_list = np.array(precision_list)
recall_list = np.array(recall_list)
f1_list = np.array(f1_list)
support_list = np.array(support_list)

# ------------------------------------------------------------
# OVERALL METRICS
# ------------------------------------------------------------

accuracy = np.mean(y_true == y_pred)

# Weighted Precision / Recall / F1
weights = support_list / support_list.sum()

weighted_precision = np.sum(
    precision_list * weights
)

weighted_recall = np.sum(
    recall_list * weights
)

weighted_f1 = np.sum(
    f1_list * weights
)

# Macro F1
macro_f1 = np.mean(f1_list)

# ------------------------------------------------------------
# TOP-3 ACCURACY
# ------------------------------------------------------------

top3_pred = np.argsort(
    test_probs,
    axis=1
)[:, -3:]

top3_correct = [
    true_label in predictions
    for true_label, predictions
    in zip(y_true, top3_pred)
]

top3_accuracy = np.mean(top3_correct)

# ------------------------------------------------------------
# PRINT TABLE 4.3
# ------------------------------------------------------------

print("=" * 65)
print("TABLE 4.3 — TEST SET RESULTS")
print("=" * 65)

print(f"Samples             : {len(y_true)}")
print(f"Correct             : {(y_true == y_pred).sum()}")
print()

print(f"Accuracy            : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision (Weighted): {weighted_precision:.4f} ({weighted_precision*100:.2f}%)")
print(f"Recall (Weighted)   : {weighted_recall:.4f} ({weighted_recall*100:.2f}%)")
print(f"F1-score (Weighted) : {weighted_f1:.4f} ({weighted_f1*100:.2f}%)")
print(f"Macro F1-score      : {macro_f1:.4f} ({macro_f1*100:.2f}%)")
print(f"Top-3 Accuracy      : {top3_accuracy:.4f} ({top3_accuracy*100:.2f}%)")

print("=" * 65)

## Deployment logic for the demo

**Face → Mouth ROI → Mouth-motion gate → collect speaking sequence → 16 frames → model → Top-3 → confidence/margin rejection.**

When mouth motion is below the calibrated threshold, the UI must stay in `WAITING / IDLE` and **must not emit a new word**. This is the key change that prevents the old behavior of continuously forcing one of 44 words.

The model still supports all **44 words**. Use `05_44word_test_class_report.csv` only to identify which words are most reliable for rehearsal; do not change the vocabulary.